## 第30章　多头架构与Block组装：从单头到完整层

> 本章目标：把第 29 章的单头复制成 8 份，拼出完整的 Transformer Block。这是"维度变换地狱"——`(B,L,D)→(B,L,H,Dh)→(B,H,L,Dh)`——写对这段代码你就超越了 80% 的初学者。掌握 FFN 的职责分离、Pre-Norm + 残差连接，最终组装出 `class TransformerBlock`。
> 前置知识：第 29 章（单头注意力）、第 21 章（LayerNorm）、第 4 章（残差连接）

> 🕵️ **开篇导语：从"孤胆侦探"到"专家陪审团"**
>
> 第 29 章，我们只派了一位侦探（单头注意力）去审讯嫌疑人。他确实能抓住某种关系——比如，他发现"苹果"和"吃"在语义上紧密相连。但语言是一头多面的巨兽：同一句话里，**语法主谓宾**、**情感色彩**（褒贬）、**指代消解**（"它"指谁）、**距离远近**（相邻词 vs 跨段词）可能同时存在。一位侦探的脑力，根本不够用。
>
> 所以，我们直接聘请一支**专家陪审团**（Multi-Head）。语法专家盯着词序结构，情感专家捕捉褒贬信号，距离专家紧盯长程依赖……**每个人独立审讯、独立打分、独立提取证词**。这带来一个棘手的问题：如何让 8 个或 12 个专家高效共用同一张"数据桌"，既不互相干扰，又能并行工作？
>
> 答案是一场精心编排的**"形状芭蕾"**（Shape Ballet）。我们用 `view` 和 `transpose` 把输入张量从 `(Batch, Seq, Dim)` 优雅地重排为 `(Batch, Head, Seq, Dim_per_Head)`，让所有头在同一时刻完成矩阵乘法，最后再拼接回统一的报告。这段代码通常不超过 10 行，但**它是全书最容易写错的地方**——而一旦你用 `assert` 锁死每一个维度变化，你对张量轴的理解将发生质变。
>
> 陪审团给出初步意见后，信息会被送入**分析室（FFN）**做深度推理——这个两层线性层占掉整个 Block 三分之二的参数量，是模型真正"存储世界知识"的部位。而为了防止层层传递导致信息失真，我们修了一条**残差高速公路**（`x + Sublayer(x)`），配合 **Pre-Norm** 确保梯度在反向传播时能一路畅行无阻直达底层。
>
> 本章结束，你将亲手 `class` 出人生第一个完整的 Transformer Block。它的核心代码不到 50 行，但从此，你眼中的"AI 黑盒"将彻底瓦解成一堆你亲手搭起来的乐高积木。



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("env ready")


### 30.1　多头拆分的"形状芭蕾" —— 全书最容易写错的地方

单头注意力的 Q、K、V 各是 `(batch, seq_len, d_k)`。多头要同时算 h 个头——每个头有自己的 Q、K、V，所有头共享输入 X，但各自有不同的投影矩阵 W_Q^head、W_K^head、W_V^head。

关键技巧：**不显式写循环**，而是把 h 个头"折叠"进张量的一个维度中，用一次矩阵乘法同时算所有头。

张量变形三部曲：
1. 投影到 `d_model` 维（Q=X@W_Q，其中 W_Q 的 shape 是 `(d_model, h*d_k)`）
2. 重塑：`(B, L, h*d_k) → (B, L, h, d_k)`
3. 转置：`(B, L, h, d_k) → (B, h, L, d_k)`

为什么必须把 Head 维度提到 Seq 前面？**为了让 `(B, h, L, d_k) @ (B, h, d_k, L)` 能并行计算所有头的注意力——每个头独立做矩阵乘法，但被 PyTorch 的批量 matmul 一次并行完成。**

📐 **定义　Multi-Head 形状变换**：`(B,L,D)→(B,L,H,Dh)→(B,H,L,Dh)`。关键：转置让 Head 和 Batch 成为前两维，Seq 和 Dh 成为矩阵乘法的两个维度。

💻 **代码　多头拆分的形状芭蕾：亲手验证每一步的 shape**


In [ ]:
import numpy as np

np.random.seed(42)
batch, seq_len, d_model, n_heads = 2, 5, 512, 8
d_k = d_model // n_heads  # 64

X = np.random.randn(batch, seq_len, d_model)

# 投影矩阵：一次性投影到 h*d_k 维
W_Q = np.random.randn(d_model, d_model) * 0.02
Q = X @ W_Q  # (2, 5, 512)

# 形状芭蕾
Q_reshaped = Q.reshape(batch, seq_len, n_heads, d_k)  # (2,5,8,64)
Q_transposed = Q_reshaped.transpose(0, 2, 1, 3)       # (2,8,5,64)

print(f"X:      {X.shape}")
print(f"Q:      {Q.shape}")
print(f"reshape: {Q_reshaped.shape} ← 把 512 拆成 8×64")
print(f"transpose: {Q_transposed.shape} ← 把 head 提到 seq 前面")
print(f"\n现在 Q[0,3] 是 batch0 第3个头对所有 token 的 query 矩阵 (5,64)")
print(f"可以直接和 K[0,3]^T (64,5) 做矩阵乘法 → (5,5) 的注意力得分")


> **关键洞察**：`transpose(0,2,1,3)` 这四个数字是全书最重要的轴变换——它把 `(Batch, Seq, Head, Dh)` 变成 `(Batch, Head, Seq, Dh)`。写错任何一个数字，后面的矩阵乘法立刻报 shape 不匹配。**用 `assert` 锁死每一步的 shape 是唯一的生存法则。**

---


### 30.2　拼接与输出投影 —— 让多头信息融合

每个头独立做完注意力后，得到 `(B, h, L, d_k)` 的输出。需要把 h 个头拼回一个大向量，再做一次线性投影融合多头信息。

步骤：`(B, h, L, d_k) → transpose → (B, L, h, d_k) → reshape → (B, L, d_model) → @ W_O → (B, L, d_model)`。

📐 **定义　输出投影**：`concat_heads @ W_O`，其中 W_O 的 shape 为 `(d_model, d_model)`。多头信息通过这个投影矩阵融合——不同头关注的不同关系（语法/语义/距离）在这里被整合。

💻 **代码　多头拼接 + 输出投影**


In [ ]:
import numpy as np

# 模拟 8 个头各自的注意力输出
batch, n_heads, seq_len, d_k = 2, 8, 5, 64
head_outputs = np.random.randn(batch, n_heads, seq_len, d_k)

# 1. 转置回 (B, L, H, Dh)
concat = head_outputs.transpose(0, 2, 1, 3)  # (2, 5, 8, 64)
# 2. 拼回 d_model
concat = concat.reshape(batch, seq_len, n_heads * d_k)  # (2, 5, 512)
# 3. 输出投影
W_O = np.random.randn(n_heads * d_k, n_heads * d_k) * 0.02
output = concat @ W_O  # (2, 5, 512)

print(f"Head outputs: {head_outputs.shape}")
print(f"After concat: {concat.shape}")
print(f"After W_O:    {output.shape} ← 和输入 X 同 shape!")


---


### 30.3　前馈网络（FFN）的职责分离

Attention 负责"从哪取信息"——聚合上下文中相关 token 的 value。FFN 负责"取到后如何理解它"——对每个 token 独立做非线性变换，存储世界知识。

结构：`Linear(d_model → 4*d_model) → GeLU → Linear(4*d_model → d_model)`。两个线性层中间夹一个激活函数。FFN 占整个 Block 约 2/3 的参数量——**模型的知识主要存储在 FFN 的权重中，而不是 Attention 中。**

📐 **定义　FFN**：`FFN(x) = GeLU(x @ W1 + b1) @ W2 + b2`。W1 (d_model, 4*d_model), W2 (4*d_model, d_model)。每个 token 独立应用相同的 FFN。

---


### 30.4　Pre-Norm + 残差连接 —— 梯度高速公路

每一个 Sublayer（Attention 或 FFN）前后各有一个关键操作：

1. **LayerNorm（Pre-Norm）**：先归一化再进入 Sublayer——`x = x + Sublayer(LayerNorm(x))`
2. **残差连接**：`x + Sublayer(...)`——第 4 章向量加法的直接应用

为什么用 Pre-Norm（先归一化再进 Sublayer）而不是 Post-Norm（先进 Sublayer 再归一化）？因为 Post-Norm 在深层网络中梯度会在归一化层被"压扁"——Pre-Norm 让梯度可以沿残差连接"高速公路"直达底层，训练更稳定。

📐 **定义　Pre-Norm 残差块**：`x = x + Sublayer(LayerNorm(x))`。LN 在 Sublayer 之前。残差连接提供梯度直达路径。

---


### 30.5　单层 Block 完整代码串讲 —— `class TransformerBlock`

把 30.1~30.4 的代码片段拼接成一个完整的 Python 类。

💻 **代码　TransformerBlock：50 行完整实现**


In [ ]:
import numpy as np

class TransformerBlock:
    def __init__(self, d_model=512, n_heads=8, d_ff=2048):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # 多头注意力投影
        self.W_Q = np.random.randn(d_model, d_model) * 0.02
        self.W_K = np.random.randn(d_model, d_model) * 0.02
        self.W_V = np.random.randn(d_model, d_model) * 0.02
        self.W_O = np.random.randn(d_model, d_model) * 0.02

        # FFN
        self.W1 = np.random.randn(d_model, d_ff) * 0.02
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.02
        self.b2 = np.zeros(d_model)

    def layer_norm(self, x, eps=1e-5):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        return (x - mean) / np.sqrt(var + eps)

    def attention(self, x):
        B, L, D = x.shape
        # 多头投影 + 形状芭蕾
        Q = x @ self.W_Q
        K = x @ self.W_K
        V = x @ self.W_V

        Q = Q.reshape(B, L, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        K = K.reshape(B, L, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        V = V.reshape(B, L, self.n_heads, self.d_k).transpose(0, 2, 1, 3)

        # 缩放点积注意力
        scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(self.d_k)
        scores = scores - scores.max(axis=-1, keepdims=True)
        attn = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
        out = attn @ V

        # 拼接 + 输出投影
        out = out.transpose(0, 2, 1, 3).reshape(B, L, D)
        return out @ self.W_O

    def ffn(self, x):
        h = np.maximum(0, x @ self.W1 + self.b1)  # ReLU activation
        return h @ self.W2 + self.b2

    def forward(self, x):
        # Pre-Norm + Attention + Residual
        x = x + self.attention(self.layer_norm(x))
        # Pre-Norm + FFN + Residual
        x = x + self.ffn(self.layer_norm(x))
        return x

# 测试
block = TransformerBlock(d_model=256, n_heads=8, d_ff=1024)
x = np.random.randn(2, 10, 256)
out = block.forward(x)
print(f"输入 shape: {x.shape}")
print(f"输出 shape: {out.shape}")
print(f"输入 == 输出 shape: {x.shape == out.shape} ← 残差连接保证 shape 不变")


> **关键洞察**：你刚刚 `class` 出了人生第一个完整的 Transformer Block。`forward` 方法只有 4 行——Pre-Norm Attention + Pre-Norm FFN——但内部包含了第 4 章的残差连接、第 5 章的点积、第 6 章的矩阵乘法、第 21 章的 LayerNorm、第 22 章的稳定 softmax、第 29 章的注意力。**这不是代码，是你前面 29 章学习的全部数学知识的一次"总装配"。**

🔗 **AI 连接**：第 31 章将在这个 Block 上跑训练循环。第 32 章将在推理时为这个 Block 加上 KV Cache。真正的 GPT-2 只是把 12 个这样的 Block 串起来，再加上 Embedding 和输出层。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. 故意把 `transpose` 写错（如忘记转置回来），观察形状报错并修正。
2. 用 `torchviz` 画 Block 的计算图，数一数残差连接产生了多少条反向传播路径。
---
> 🔗 **章末钩子**：我们有了一个 Block，但它只会做"特征提取"。怎么让它像 ChatGPT 一样**一个字一个字地往外蹦**？训练和推理的逻辑有何不同？→ 引向第 31 章。
> 预览：下一章——**训练循环的微观解剖**，5 行代码背后的 10 个陷阱。
